Membuat SparkSession
-

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, TimestampType
)
from pyspark.sql.functions import (
    col, sum as spark_sum, count, window
)

import os
import shutil
import time
import pandas as pd
from datetime import datetime

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Mengatur jumlah partisi shuffle
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession berhasil dibuat.")
print("Versi Spark:", spark.version)

26/11/05 13:59:35 WARN Utils: Your hostname, zaydann resolves to a loopback address: 127.0.1.1; using 192.168.50.46 instead (on interface wlo1)
26/11/05 13:59:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 13:59:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat.
Versi Spark: 3.5.9


A. Menyiapkan Streaming Source
-

In [4]:
# Mendefinisikan skema data pesanan
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

print("Skema data pesanan berhasil dibuat.")
print(skema_pesanan)

Skema data pesanan berhasil dibuat.
StructType([StructField('pesanan_id', StringType(), True), StructField('nama_cabang', StringType(), True), StructField('nama_menu', StringType(), True), StructField('qty', IntegerType(), True), StructField('harga_satuan', IntegerType(), True), StructField('waktu_pesan', TimestampType(), True)])


In [5]:
# Membuat folder simulasi streaming
folder_tugas = "tugas10_stream"

# Membersihkan folder jika sebelumnya sudah ada
shutil.rmtree(folder_tugas, ignore_errors=True)

# Membuat folder baru
os.makedirs(folder_tugas, exist_ok=True)

print("Folder streaming berhasil dibuat:", folder_tugas)

Folder streaming berhasil dibuat: tugas10_stream


In [6]:
# Membaca data dari folder menggunakan readStream
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

print("Streaming DataFrame berhasil dibuat.")
print("Apakah termasuk streaming:", df_stream.isStreaming)

df_stream.printSchema()

Streaming DataFrame berhasil dibuat.
Apakah termasuk streaming: True
root
 |-- pesanan_id: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- waktu_pesan: timestamp (nullable = true)



B. Agregasi per Cabang
-

In [7]:
# Menambahkan kolom total_harga
df_pesanan = df_stream.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

# Mengelompokkan data berdasarkan nama cabang
agregasi_cabang = df_pesanan.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_harga"),
    count("pesanan_id").alias("jumlah_pesanan")
)

print("Agregasi per cabang berhasil dibuat.")

Agregasi per cabang berhasil dibuat.


In [8]:
# Menjalankan streaming query per cabang
query1 = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming query agregasi cabang berhasil dijalankan.")
print("Status aktif:", query1.isActive)

Streaming query agregasi cabang berhasil dijalankan.
Status aktif: True


C. Agregasi Berbasis Window Waktu
-

In [9]:
# Membaca kembali data streaming dari folder yang sama
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

# Menambahkan kolom total_harga
df_pesanan_waktu = df_stream_waktu.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

# Mengelompokkan data berdasarkan window waktu 2 menit
agregasi_waktu = df_pesanan_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_harga")
)

print("Agregasi window waktu berhasil dibuat.")

Agregasi window waktu berhasil dibuat.


In [10]:
# Menjalankan streaming query window waktu
query2 = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming query window waktu berhasil dijalankan.")
print("Status aktif:", query2.isActive)

Streaming query window waktu berhasil dijalankan.
Status aktif: True


D. Menguji dengan Data Simulasi
-

Batch 1 — Data Pesanan Pertama

In [11]:
# Membuat waktu acuan untuk simulasi
waktu_acuan = datetime.now()

# Membuat data batch pertama
batch_1 = pd.DataFrame({
    "pesanan_id": ["P001", "P002", "P003"],
    "nama_cabang": ["Magelang", "Solo", "Magelang"],
    "nama_menu": ["Kopi Susu", "Cappuccino", "Americano"],
    "qty": [2, 3, 1],
    "harga_satuan": [18000, 22000, 15000],
    "waktu_pesan": [
        waktu_acuan.strftime("%Y-%m-%d %H:%M:%S")
    ] * 3
})

# Menyimpan batch pertama ke folder streaming
batch_1.to_csv(
    f"{folder_tugas}/batch_1.csv",
    index=False
)

print("Batch 1 berhasil dikirim.")

# Menunggu Spark memproses data
time.sleep(5)
query1.processAllAvailable()
query2.processAllAvailable()

# Menampilkan hasil agregasi per cabang
spark.sql("""
    SELECT *
    FROM agregasi_cabang
    ORDER BY total_harga DESC
""").show()

Batch 1 berhasil dikirim.


+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|       Solo|      66000|             1|
|   Magelang|      51000|             2|
+-----------+-----------+--------------+



Batch 2 — Data Pesanan Kedua

In [12]:
# Membuat data batch kedua
batch_2 = pd.DataFrame({
    "pesanan_id": ["P004", "P005", "P006"],
    "nama_cabang": ["Yogyakarta", "Magelang", "Solo"],
    "nama_menu": ["Cappuccino", "Kopi Susu", "Americano"],
    "qty": [2, 2, 4],
    "harga_satuan": [25000, 22000, 15000],
    "waktu_pesan": [
        waktu_acuan.strftime("%Y-%m-%d %H:%M:%S")
    ] * 3
})

# Menyimpan batch kedua
batch_2.to_csv(
    f"{folder_tugas}/batch_2.csv",
    index=False
)

print("Batch 2 berhasil dikirim.")

# Menunggu Spark memproses data
time.sleep(5)
query1.processAllAvailable()
query2.processAllAvailable()

# Menampilkan hasil agregasi terbaru
spark.sql("""
    SELECT *
    FROM agregasi_cabang
    ORDER BY total_harga DESC
""").show()

Batch 2 berhasil dikirim.
+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|       Solo|     126000|             2|
|   Magelang|      95000|             3|
| Yogyakarta|      50000|             1|
+-----------+-----------+--------------+



Batch 3 — Data Pesanan Ketiga

In [13]:
# Membuat data batch ketiga
batch_3 = pd.DataFrame({
    "pesanan_id": ["P007", "P008", "P009"],
    "nama_cabang": ["Yogyakarta", "Semarang", "Magelang"],
    "nama_menu": ["Kopi Susu", "Cappuccino", "Cappuccino"],
    "qty": [3, 2, 1],
    "harga_satuan": [18000, 22000, 25000],
    "waktu_pesan": [
        waktu_acuan.strftime("%Y-%m-%d %H:%M:%S")
    ] * 3
})

# Menyimpan batch ketiga
batch_3.to_csv(
    f"{folder_tugas}/batch_3.csv",
    index=False
)

print("Batch 3 berhasil dikirim.")

# Menunggu Spark memproses data
time.sleep(5)
query1.processAllAvailable()
query2.processAllAvailable()

# Menampilkan hasil agregasi terbaru
spark.sql("""
    SELECT *
    FROM agregasi_cabang
    ORDER BY total_harga DESC
""").show()

Batch 3 berhasil dikirim.
+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|       Solo|     126000|             2|
|   Magelang|     120000|             4|
| Yogyakarta|     104000|             2|
|   Semarang|      44000|             1|
+-----------+-----------+--------------+



Melihat Hasil Agregasi Window Waktu

In [14]:
# Menampilkan hasil agregasi berdasarkan window waktu
spark.sql("""
    SELECT
        window.start AS waktu_mulai,
        window.end AS waktu_selesai,
        total_harga
    FROM agregasi_waktu
""").show(truncate=False)

+-------------------+-------------------+-----------+
|waktu_mulai        |waktu_selesai      |total_harga|
+-------------------+-------------------+-----------+
|2026-11-05 14:00:00|2026-11-05 14:02:00|394000     |
+-------------------+-------------------+-----------+



E. Sink Berbasis Berkas (Tantangan)
-

1. Membuat Streaming Query Ketiga

In [15]:
# Membersihkan folder output dan checkpoint jika sudah ada
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)

# Menjalankan streaming query ke format Parquet
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Streaming query Parquet berhasil dijalankan.")
print("Status aktif:", query3.isActive)

Streaming query Parquet berhasil dijalankan.
Status aktif: True


2. Mengirim Batch Data Baru

In [16]:
# Membuat data batch keempat
batch_4 = pd.DataFrame({
    "pesanan_id": ["P010", "P011", "P012"],
    "nama_cabang": ["Solo", "Magelang", "Yogyakarta"],
    "nama_menu": ["Kopi Susu", "Teh", "Roti"],
    "qty": [2, 1, 2],
    "harga_satuan": [20000, 12000, 10000],
    "waktu_pesan": [
        datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    ] * 3
})

# Menyimpan batch keempat
batch_4.to_csv(
    f"{folder_tugas}/batch_4.csv",
    index=False
)

print("Batch keempat berhasil dikirim.")

# Menunggu Spark memproses data
time.sleep(5)
query3.processAllAvailable()

print("Batch berhasil diproses ke Parquet.")

Batch keempat berhasil dikirim.
Batch berhasil diproses ke Parquet.


3. Memeriksa File Parquet

In [17]:
# Melihat isi folder output Parquet
!ls -lh tugas10_output

total 20K
-rw-r--r-- 1 zaydann zaydann 1,9K Nov  5 14:00 part-00000-3e5aa805-2815-4b0c-becc-c9a770098283-c000.snappy.parquet
-rw-r--r-- 1 zaydann zaydann 1,9K Nov  5 14:00 part-00000-e755492d-e156-4e56-b1d8-ae44cbb8a8ca-c000.snappy.parquet
-rw-r--r-- 1 zaydann zaydann 1,9K Nov  5 14:00 part-00001-3e92b82d-e34b-4395-8a98-589f9676d631-c000.snappy.parquet
-rw-r--r-- 1 zaydann zaydann 1,9K Nov  5 14:00 part-00002-8efc8e56-cb7f-4f26-ac88-97741c182e06-c000.snappy.parquet
drwxr-xr-x 2 zaydann zaydann 4,0K Nov  5 14:00 _spark_metadata


4. Membaca Kembali File Parquet

In [18]:
# Membaca file Parquet menggunakan Spark
df_hasil_parquet = spark.read.parquet("tugas10_output")

# Menampilkan data yang berhasil dibaca
df_hasil_parquet.show(truncate=False)

# Menghitung jumlah data
print("Jumlah data:", df_hasil_parquet.count())

+----------+-----------+----------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu |qty|harga_satuan|waktu_pesan        |
+----------+-----------+----------+---+------------+-------------------+
|P004      |Yogyakarta |Cappuccino|2  |25000       |2026-11-05 14:00:04|
|P005      |Magelang   |Kopi Susu |2  |22000       |2026-11-05 14:00:04|
|P006      |Solo       |Americano |4  |15000       |2026-11-05 14:00:04|
|P007      |Yogyakarta |Kopi Susu |3  |18000       |2026-11-05 14:00:04|
|P008      |Semarang   |Cappuccino|2  |22000       |2026-11-05 14:00:04|
|P009      |Magelang   |Cappuccino|1  |25000       |2026-11-05 14:00:04|
|P001      |Magelang   |Kopi Susu |2  |18000       |2026-11-05 14:00:04|
|P002      |Solo       |Cappuccino|3  |22000       |2026-11-05 14:00:04|
|P003      |Magelang   |Americano |1  |15000       |2026-11-05 14:00:04|
|P010      |Solo       |Kopi Susu |2  |20000       |2026-11-05 14:00:53|
|P011      |Magelang   |Teh       |1  |12000       

Menghentikan Seluruh Streaming Query dan SparkSession
-

In [19]:
# Menghentikan seluruh streaming query
query1.stop()
query2.stop()
query3.stop()

print("Seluruh streaming query berhasil dihentikan.")

# Menutup SparkSession
spark.stop()

print("SparkSession berhasil ditutup.")

Seluruh streaming query berhasil dihentikan.
SparkSession berhasil ditutup.
